# CELL 1

In [14]:
import numpy as np
from plasmapy.formulary import Debye_length
from astropy import units as u

# ── Physical Constants (SI) ───────────────────────────────────
e      = 1.602176634e-19   # [C]        elementary charge
eps0   = 8.8541878128e-12  # [F/m]      vacuum permittivity
m_p    = 1.67262192e-27    # [kg]       proton mass
k_B    = 1.380649e-23      # [J/K]      Boltzmann constant
N_A    = 6.02214076e23     # [mol⁻¹]   Avogadro's number

# ── Plasma Conditions ─────────────────────────────────────────
T_eV   = 0.1               # [eV]       electron temperature
T_K    = T_eV * 11604.52   # [K]        electron temperature
n_e    = 1e21              # [m⁻³]      number density

# ── Derived Parameters via PlasmaPy ───────────────────────────
lambda_D_m = Debye_length(T_eV * u.eV, n_e * u.m**-3).to(u.m).value  # [m]
lambda_D_A = lambda_D_m * 1e10                                          # [Å]

# ── Normalization Scales ───────────────────────────────────────
# Simulation runs in normalized units — see table below for SI conversion
energy_scale = k_B * T_K              # [J]   1 energy unit = k_BT
time_scale   = np.sqrt(m_p * lambda_D_m**2 / (k_B * T_K))  # [s]  1 τ

# ── Yukawa Prefactor ───────────────────────────────────────────
# V(r) = A_norm * exp(-κr) / r   [k_BT·λ_D]
# A_norm = e²/4πε₀ / (k_BT · λ_D)  [dimensionless]
A_coulomb = e**2 / (4 * np.pi * eps0)              # [J·m]
A_norm    = A_coulomb / (k_B * T_K * lambda_D_m)   # [dimensionless]

# ── Coupling Parameter ─────────────────────────────────────────
N_protons  = 150
box_size   = 10.0               # [λ_D]  simulation box side
kappa_norm = 1.0                # [1/λ_D] screening parameter (= 1 by definition)
cutoff     = 3.0                # [λ_D]  Yukawa cutoff (negligible beyond 3 λ_D)
damping    = 0.1                # [τ]    NVT thermostat damping
steps      = 5000               # [steps] production run
dump_freq  = 25                 # [steps] output frequency → 200 frames
timestep   = 0.001              # [τ]    production timestep

r_avg  = (3 / (4 * np.pi * (N_protons / box_size**3))) ** (1/3)  # [λ_D] Wigner-Seitz radius
Gamma  = A_norm / r_avg                                            # [dimensionless] coupling

print("=" * 60)
print("  PLASMA SCREENING — UNIT REFERENCE")
print("=" * 60)
print(f"\n[Normalization Basis]")
print(f"  Length  (1 λ_D)  : {lambda_D_m:.4e} m  =  {lambda_D_A:.4f} Å")
print(f"  Energy  (1 k_BT) : {energy_scale:.4e} J  =  {energy_scale/e:.6f} eV")
print(f"  Mass    (1 m_p)  : {m_p:.4e} kg  =  1.007 amu")
print(f"  Time    (1 τ)    : {time_scale:.4e} s  =  {time_scale*1e12:.4f} ps")
print(f"\n[Plasma Conditions]")
print(f"  Temperature      : {T_eV} eV  =  {T_K:.2f} K")
print(f"  Number Density   : {n_e:.2e} m⁻³")
print(f"  Debye Length λ_D : {lambda_D_m:.4e} m  =  {lambda_D_A:.4f} Å")
print(f"\n[Simulation Parameters]")
print(f"  Protons          : {N_protons}")
print(f"  Box Side         : {box_size} λ_D  =  {box_size*lambda_D_A:.2f} Å")
print(f"  Yukawa κ         : {kappa_norm} λ_D⁻¹  (= 1/λ_D by definition)")
print(f"  Yukawa A (norm)  : {A_norm:.6f}  [= e²/4πε₀k_BTλ_D]")
print(f"  Cutoff           : {cutoff} λ_D  =  {cutoff*lambda_D_A:.2f} Å")
print(f"  Coupling Γ       : {Gamma:.5f}  ({'weakly coupled Γ<<1' if Gamma < 1 else 'strongly coupled Γ>>1'})")
print(f"  Timestep         : {timestep} τ  =  {timestep*time_scale*1e15:.4f} fs")
print(f"  Total Sim Time   : {steps*timestep*time_scale*1e12:.4f} ps")
print(f"  Frames           : {steps // dump_freq}")

  PLASMA SCREENING — UNIT REFERENCE

[Normalization Basis]
  Length  (1 λ_D)  : 7.4339e-08 m  =  743.3942 Å
  Energy  (1 k_BT) : 1.6022e-20 J  =  0.100000 eV
  Mass    (1 m_p)  : 1.6726e-27 kg  =  1.007 amu
  Time    (1 τ)    : 2.4019e-11 s  =  24.0194 ps

[Plasma Conditions]
  Temperature      : 0.1 eV  =  1160.45 K
  Number Density   : 1.00e+21 m⁻³
  Debye Length λ_D : 7.4339e-08 m  =  743.3942 Å

[Simulation Parameters]
  Protons          : 150
  Box Side         : 10.0 λ_D  =  7433.94 Å
  Yukawa κ         : 1.0 λ_D⁻¹  (= 1/λ_D by definition)
  Yukawa A (norm)  : 0.193701  [= e²/4πε₀k_BTλ_D]
  Cutoff           : 3.0 λ_D  =  2230.18 Å
  Coupling Γ       : 0.16590  (weakly coupled Γ<<1)
  Timestep         : 0.001 τ  =  24.0194 fs
  Total Sim Time   : 120.0972 ps
  Frames           : 200


# CELL 2

In [15]:
from lammps import lammps

lmp = lammps()

lmp.commands_string(f"""
# ══════════════════════════════════════════════════════════════════
#  PLASMA SCREENING — YUKAWA (DEBYE-HÜCKEL) SIMULATION
#
#  Unit system: normalized (dimensionless LJ-style)
#  See Cell 1 for full SI conversion table.
#
#  Normalization basis:
#    Length  → λ_D  = {lambda_D_m:.4e} m  =  {lambda_D_A:.4f} Å
#    Energy  → k_BT = {energy_scale:.4e} J  =  {energy_scale/e:.6f} eV
#    Mass    → m_p  = {m_p:.4e} kg
#    Time    → τ    = {time_scale:.4e} s  =  {time_scale*1e12:.4f} ps
#
#  Yukawa potential: V(r) = A·exp(-κr)/r  [k_BT]
#    A = {A_norm:.6f}  [= e²/4πε₀k_BTλ_D, dimensionless]
#    κ = {kappa_norm}  [λ_D⁻¹, = 1/λ_D by normalization]
#  Electrons implicit — encoded in κ (Debye-Hückel theory)
# ══════════════════════════════════════════════════════════════════

# ── Unit System ────────────────────────────────────────────────────
units         lj                         # normalized (see Cell 1 for SI conversion)
atom_style    atomic                     # id type x[λ_D] y[λ_D] z[λ_D]
boundary      p p p                      # periodic boundaries x, y, z

# ── Simulation Box ─────────────────────────────────────────────────
# {box_size} λ_D = {box_size*lambda_D_A:.2f} Å = {box_size*lambda_D_m*1e9:.4f} nm per side
region        box block 0 {box_size} 0 {box_size} 0 {box_size}  # [λ_D]
create_box    1 box                      # 1 atom type: protons only
create_atoms  1 random {N_protons} 12345 box  # {N_protons} protons, random placement

# ── Mass [m_p] ─────────────────────────────────────────────────────
mass          1 1.0                      # [m_p] proton mass (= {m_p:.4e} kg = 1.007 amu)

# ── Yukawa (Debye-Hückel) Screened Potential ───────────────────────
# V(r) = A·exp(-κ·r)/r  [k_BT]
# Electrons are implicit — their screening effect is fully encoded
# in κ = 1/λ_D. This is exact for weakly coupled plasmas (Γ << 1).
# κ = {kappa_norm} [λ_D⁻¹]  =  {kappa_norm/lambda_D_m:.4e} m⁻¹
# A = {A_norm:.6f} [dimensionless]  =  e²/4πε₀k_BTλ_D
# cutoff = {cutoff} [λ_D]  =  {cutoff*lambda_D_A:.2f} Å
# (Yukawa decays as e^-3 ≈ 0.05 at r=3λ_D — negligible beyond cutoff)
pair_style    yukawa {kappa_norm} {cutoff}  # κ[λ_D⁻¹]  cutoff[λ_D]
pair_coeff    1 1 {A_norm:.6f}             # A[dimensionless] = e²/4πε₀k_BTλ_D

# ── Neighbor List ──────────────────────────────────────────────────
neigh_modify  one 5000 delay 0 every 1 check yes  # max 5000 neighbors, check every step

# ── Initial Velocities ─────────────────────────────────────────────
# Maxwell-Boltzmann distribution at T = 1.0 k_BT/k_B
velocity      all create 1.0 54321 dist gaussian  # [k_BT/k_B] target temperature

# ── NVT Thermostat ─────────────────────────────────────────────────
# Nosé-Hoover thermostat maintains constant temperature
fix           1 all nvt temp 1.0 1.0 {damping}  # T_start[k_BT/k_B] T_end[k_BT/k_B] τ_damp[τ]

# ── Dump Positions ─────────────────────────────────────────────────
dump          1 all custom {dump_freq} plasma.dump id x y z  # positions [λ_D]
dump_modify   1 sort id                                       # sort by atom id

# ── Thermo Output ──────────────────────────────────────────────────
# T[k_BT/k_B] PE[k_BT] KE[k_BT] E_total[k_BT] P[k_BT/λ_D³]
thermo_style  custom step temp pe ke etotal press
thermo        {dump_freq}               # print every {dump_freq} steps

# ── Production Run ─────────────────────────────────────────────────
timestep      {timestep}               # [τ] = {timestep*time_scale*1e15:.4f} fs
run           {steps}                  # {steps} steps = {steps*timestep*time_scale*1e12:.4f} ps
""")

lmp.close()
print(f"✓ LAMMPS simulation complete")
print(f"  Timestep             : {timestep} τ  =  {timestep*time_scale*1e15:.4f} fs")
print(f"  Total simulated time : {steps*timestep*time_scale*1e12:.4f} ps")
print(f"  Frames written       : {steps // dump_freq}")

LAMMPS (29 Aug 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Created orthogonal box = (0 0 0) to (10 10 10)
  1 by 1 by 1 MPI processor grid
Created 150 atoms
  using lattice units in orthogonal box = (0 0 0) to (10 10 10)
  create_atoms CPU = 0.000 seconds
Generated 0 of 0 mixed pair_coeff terms from geometric mixing rule
Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 5000, page size: 100000
  master list distance cutoff = 3.3
  ghost atom cutoff = 3.3
  binsize = 1.65, bins = 7 7 7
  1 neighbor lists, perpetual/occasional/extra = 1 0 0
  (1) pair yukawa, perpetual
      attributes: half, newton on
      pair build: half/bin/atomonly/newton
      stencil: half/bin/3d
      bin: standard
Setting up Verlet run ...
  Unit style    : lj
  Current step  : 0
  Time step     : 0.001
Per MPI rank memory allocation (min/avg/max) = 3.068 | 3.068 | 3.068 Mbytes


# CELL 3

In [16]:
import numpy as np

def parse_lammps_dump(filepath):
    """Read LAMMPS dump file → list of (N,3) position arrays [λ_D]."""
    frames = []
    with open(filepath, "r") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if "ITEM: TIMESTEP" in lines[i]:
            i += 2  # skip timestep value
            i += 2  # skip NUMBER OF ATOMS header + value
            i += 4  # skip BOX BOUNDS header + 3 bound lines
            i += 1  # skip ITEM: ATOMS header

            positions = []
            while i < len(lines) and "ITEM:" not in lines[i]:
                parts = lines[i].split()
                x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
                positions.append([x, y, z])
                i += 1
            frames.append(np.array(positions))
        else:
            i += 1
    return frames

frames = parse_lammps_dump("plasma.dump")

print(f"✓ Parsed {len(frames)} frames")
print(f"✓ Protons per frame : {len(frames[0])}")
print(f"✓ Positions in λ_D units (box = 0 → {box_size} λ_D)")
print(f"\nSample position ranges (frame 1):")
pos0 = frames[1]
print(f"  x: {pos0[:,0].min():.3f} → {pos0[:,0].max():.3f} λ_D")
print(f"  y: {pos0[:,1].min():.3f} → {pos0[:,1].max():.3f} λ_D")
print(f"  z: {pos0[:,2].min():.3f} → {pos0[:,2].max():.3f} λ_D")

✓ Parsed 201 frames
✓ Protons per frame : 150
✓ Positions in λ_D units (box = 0 → 10.0 λ_D)

Sample position ranges (frame 1):
  x: 0.034 → 9.902 λ_D
  y: 0.174 → 9.815 λ_D
  z: 0.013 → 9.985 λ_D


# CELL 4

In [17]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
from joblib import Parallel, delayed
from multiprocessing import cpu_count
import numpy as np

# ── Skip frame 0 (pre-run initial state) ──────────────────────
frames_use = frames[1:]
n_frames   = len(frames_use)

# ── RDF Setup ─────────────────────────────────────────────────
n_bins    = 80
r_max     = cutoff * 0.9                         # [λ_D] stay within cutoff
r_bins    = np.linspace(0.05, r_max, n_bins+1)   # [λ_D]
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])    # [λ_D] bin centers

# ── Analytical Debye Cloud Setup ───────────────────────────────
ring_res  = 300                                  # image resolution [pixels]
ring_half = ring_res // 2

# ── Minimum Image Convention (periodic boundary) ───────────────
def min_image(delta, box):
    """Wrap pairwise distances for periodic boundaries. [λ_D]"""
    return delta - box * np.round(delta / box)

# ── Vectorized RDF for one frame ──────────────────────────────
def compute_frame(args):
    """Fully vectorized proton-proton RDF for one frame."""
    frame_idx, pos = args
    n_atoms     = len(pos)
    n_density   = n_atoms / box_size**3          # [λ_D⁻³] number density

    # Vectorized pairwise distances (N×N×3) → (N×N) [λ_D]
    dr    = pos[:, np.newaxis, :] - pos[np.newaxis, :, :]   # [λ_D]
    dr   -= box_size * np.round(dr / box_size)               # minimum image [λ_D]
    dists = np.linalg.norm(dr, axis=2)                       # [λ_D]

    # Upper triangle only (avoid double counting)
    idx_i, idx_j = np.triu_indices(n_atoms, k=1)
    dists_flat   = dists[idx_i, idx_j]

    # Filter valid range
    mask        = (dists_flat > 0.05) & (dists_flat < r_max)
    dists_valid = dists_flat[mask]

    # RDF histogram
    hist, _ = np.histogram(dists_valid, bins=r_bins)         # [counts]

    # Normalize: g(r) = histogram / (N × shell_volume × density)
    shell_vols = (4/3) * np.pi * (r_bins[1:]**3 - r_bins[:-1]**3)  # [λ_D³]
    expected   = n_density * shell_vols * n_atoms / 2               # [counts]
    rdf        = np.where(expected > 0, hist / expected, 0)         # [dimensionless]

    return frame_idx, rdf.astype(np.float64)

# ── Parallel RDF Precomputation ────────────────────────────────
n_cores = cpu_count()
print(f"Precomputing RDF across {n_frames} frames using {n_cores} cores...")
args    = [(i, frames_use[i]) for i in range(n_frames)]

results = Parallel(n_jobs=n_cores, prefer="threads")(
    delayed(compute_frame)(arg) for arg in args
)
results.sort(key=lambda x: x[0])

# Build cumulative RDF history
rdf_history = []
rdf_accum   = np.zeros(n_bins)
for i, (_, rdf) in enumerate(results):
    rdf_accum += rdf
    rdf_history.append(rdf_accum.copy() / (i + 1))

print(f"✓ RDF computed across {n_frames} frames")

# ── Analytical Debye-Hückel Cloud ─────────────────────────────
# Linearized Debye-Hückel electron density around a proton:
# n_e(r)/n_0 = 1 + A_norm·exp(-κ·r)/r   [dimensionless]
# This is the exact theoretical prediction for weakly coupled
# plasmas (Γ << 1) — shows the electron screening cloud
yy, xx  = np.mgrid[-ring_half:ring_half, -ring_half:ring_half]
r_grid  = np.sqrt(xx**2 + yy**2) / ring_half * r_max  # [λ_D]
r_safe  = np.where(r_grid > 0.05, r_grid, 0.05)        # [λ_D] avoid r=0 singularity

# Electron density relative to background [dimensionless]
phi_r            = A_norm * np.exp(-kappa_norm * r_safe) / r_safe  # [k_BT] Yukawa φ(r)
n_e_norm         = 1.0 + phi_r                                      # [dimensionless] n_e(r)/n_0
n_e_norm[r_grid >= r_max]  = 0.0                                    # mask outside cutoff
n_e_norm[r_grid < 0.05]    = 0.0                                    # mask r=0 singularity
cloud_smooth     = gaussian_filter(n_e_norm, sigma=3)               # smooth for display

print(f"✓ Analytical Debye cloud computed")
print(f"  Peak n_e(r)/n_0  : {cloud_smooth.max():.2f}  (at r → 0)")
print(f"  Background n_e/n_0: 1.0  (at r >> λ_D)")

# ── Build Figure Layout ────────────────────────────────────────
fig      = plt.figure(figsize=(15, 10), facecolor="#0a0a1a")
outer_gs = gridspec.GridSpec(2, 1, height_ratios=[5, 1],
                             hspace=0.08, figure=fig)
top_gs   = gridspec.GridSpecFromSubplotSpec(2, 2,
                                            subplot_spec=outer_gs[0],
                                            width_ratios=[1.3, 1],
                                            hspace=0.45, wspace=0.35)

ax3d     = fig.add_subplot(top_gs[:, 0], projection='3d')
ax_rdf   = fig.add_subplot(top_gs[0, 1])
ax_ring  = fig.add_subplot(top_gs[1, 1])
ax_stats = fig.add_subplot(outer_gs[1])
ax_stats.set_facecolor("#0d0d1f")
ax_stats.axis('off')

for ax in [ax_rdf, ax_ring]:
    ax.set_facecolor("#0d0d2b")
    for spine in ax.spines.values():
        spine.set_edgecolor('#444466')
    ax.tick_params(colors='#aaaacc', labelsize=7)
    ax.xaxis.label.set_color('#aaaacc')
    ax.yaxis.label.set_color('#aaaacc')
    ax.title.set_color('white')

# ── Initialize Analytical Cloud Image ─────────────────────────
ring_img = ax_ring.imshow(
    cloud_smooth,
    extent=[-r_max, r_max, -r_max, r_max],
    origin='lower', cmap='inferno', aspect='equal',
    vmin=0, vmax=cloud_smooth.max()
)
theta = np.linspace(0, 2*np.pi, 300)
ax_ring.plot(np.cos(theta), np.sin(theta),
             color='cyan', linewidth=0.8, linestyle='--',
             alpha=0.6, label='r = 1 λ_D')
ax_ring.legend(fontsize=6, facecolor='#1a1a2e',
               labelcolor='white', framealpha=0.7, loc='upper right')

cbar = fig.colorbar(ring_img, ax=ax_ring, fraction=0.046, pad=0.04)
cbar.ax.tick_params(colors='#aaaacc', labelsize=6)
cbar.set_label('$n_e(r)/n_0$', color='#aaaacc', fontsize=7)

# ── Stats Panel ───────────────────────────────────────────────
col_style    = dict(color='#aaaacc', fontsize=8, fontfamily='monospace',
                    transform=ax_stats.transAxes, va='center')
header_style = dict(color='white', fontsize=8, fontweight='bold',
                    fontfamily='monospace', transform=ax_stats.transAxes,
                    va='center')

ax_stats.axhline(y=0.95, color='#334477', linewidth=0.8)

ax_stats.text(0.01, 0.72, "── PARTICLES ──",        **header_style)
ax_stats.text(0.21, 0.72, "── PLASMA CONDITIONS ──", **header_style)
ax_stats.text(0.50, 0.72, "── DEBYE PARAMETERS ──",  **header_style)
ax_stats.text(0.74, 0.72, "── SIMULATION ──",        **header_style)

ax_stats.text(0.01, 0.45, f"Protons   : {N_protons}",                               **col_style)
ax_stats.text(0.01, 0.20, f"Electrons : implicit (κ = 1/λ_D)",                      **col_style)
ax_stats.text(0.21, 0.45, f"Temperature : {T_eV} eV  ({T_K:.0f} K)",               **col_style)
ax_stats.text(0.21, 0.20, f"Density     : {n_e:.2e} m⁻³",                          **col_style)
ax_stats.text(0.50, 0.45, f"Debye Length : {lambda_D_m:.4e} m",                    **col_style)
ax_stats.text(0.50, 0.20, f"Coupling Γ   : {Gamma:.5f}  (weakly coupled)"
              if Gamma < 1 else
              f"Coupling Γ   : {Gamma:.5f}  (strongly coupled)",                    **col_style)
ax_stats.text(0.74, 0.45, f"MD Steps  : {steps}  |  Timestep : {timestep} τ",      **col_style)
ax_stats.text(0.74, 0.20, f"Box Size  : {box_size} λ_D  |  Cutoff : {cutoff} λ_D", **col_style)

rdf_peak_text = ax_stats.text(0.50, -0.15, "", fontsize=8, color='mediumpurple',
                               fontfamily='monospace', transform=ax_stats.transAxes,
                               va='center', ha='center', clip_on=False)

# ── Animation Update ──────────────────────────────────────────
def update(frame_idx):
    # ── Left: 3D proton positions ──────────────────────────────
    ax3d.cla()
    ax3d.set_facecolor("#0a0a1a")
    pos = frames_use[frame_idx]
    ax3d.scatter(pos[:,0], pos[:,1], pos[:,2],
                 c='crimson', s=40, alpha=0.9,
                 edgecolors='darkred', linewidths=0.3)
    ax3d.set_xlim(0, box_size)
    ax3d.set_ylim(0, box_size)
    ax3d.set_zlim(0, box_size)
    ax3d.set_xlabel("x (λ_D)", color='white', fontsize=7)
    ax3d.set_ylabel("y (λ_D)", color='white', fontsize=7)
    ax3d.set_zlabel("z (λ_D)", color='white', fontsize=7)
    ax3d.tick_params(colors='white', labelsize=6)
    ax3d.set_title(f"Proton Positions  |  {N_protons} protons\n"
                   f"Frame {frame_idx+1}/{n_frames}",
                   color='white', fontsize=9)

    # ── Right top: Running proton-proton RDF ───────────────────
    ax_rdf.cla()
    ax_rdf.set_facecolor("#0d0d2b")
    rdf = rdf_history[frame_idx]
    ax_rdf.plot(r_centers, rdf, color='mediumpurple', linewidth=1.5)
    ax_rdf.axhline(y=1.0, color='gray', linestyle='--',
                   linewidth=0.8, alpha=0.6, label='Random (g=1)')
    ax_rdf.fill_between(r_centers, 1.0, rdf, where=(rdf > 1),
                        alpha=0.25, color='mediumpurple', label='Proton excess')
    ax_rdf.fill_between(r_centers, 1.0, rdf, where=(rdf < 1),
                        alpha=0.25, color='tomato',      label='Proton deficit')
    ax_rdf.set_xlim(0, r_max)
    ax_rdf.set_ylim(0, max(2.5, rdf.max()*1.1))
    ax_rdf.set_xlabel("r (λ_D)", color='#aaaacc', fontsize=8)
    ax_rdf.set_ylabel("g(r)  proton-proton", color='#aaaacc', fontsize=8)
    ax_rdf.set_title("Radial Distribution Function\n(accumulating over time)",
                     color='white', fontsize=8)
    ax_rdf.legend(fontsize=6, facecolor='#1a1a2e',
                  labelcolor='white', framealpha=0.7)
    ax_rdf.tick_params(colors='#aaaacc', labelsize=7)
    for spine in ax_rdf.spines.values():
        spine.set_edgecolor('#444466')

    # ── Right bottom: Analytical Debye cloud (static) ──────────
    # n_e(r)/n_0 = 1 + A·exp(-κr)/r  [linearized Debye-Hückel]
    # Bright center = high electron density near proton
    # Fades to uniform (n_e/n_0 = 1) beyond r = 1 λ_D
    ring_img.set_data(cloud_smooth)
    ax_ring.set_xlabel("Δx (λ_D)", color='#aaaacc', fontsize=8)
    ax_ring.set_ylabel("Δy (λ_D)", color='#aaaacc', fontsize=8)
    ax_ring.set_title(
        "Debye Screening Cloud  (analytical)\n"
        r"$n_e(r)/n_0 = 1 + A \cdot e^{-\kappa r}/r$",
        color='white', fontsize=8)

    # ── Live stats ─────────────────────────────────────────────
    current_peak   = rdf_history[frame_idx].max()
    current_peak_r = r_centers[np.argmax(rdf_history[frame_idx])]
    rdf_peak_text.set_text(
        f"[ Live ]  RDF Peak: g(r) = {current_peak:.3f}  |  "
        f"Peak at r = {current_peak_r:.3f} λ_D  |  "
        f"Frame {frame_idx+1}/{n_frames}"
    )

    fig.patch.set_facecolor("#0a0a1a")

ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=60)

writer = animation.FFMpegWriter(fps=20, bitrate=2400)
ani.save("plasma_screening.mp4", writer=writer)
plt.close()
print("✓ Saved plasma_screening.mp4")

Precomputing RDF across 200 frames using 10 cores...
✓ RDF computed across 200 frames
✓ Analytical Debye cloud computed
  Peak n_e(r)/n_0  : 2.47  (at r → 0)
  Background n_e/n_0: 1.0  (at r >> λ_D)
✓ Saved plasma_screening.mp4
